# Support Vector Machine (SVM) Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Crucial for SVM)

**Important:** SVM is sensitive to feature scaling. Features must be scaled to similar ranges for optimal performance.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Support Vector Machine (SVM) Regression Formula and Concepts

**SVM Regression (SVR) Objective:**

Unlike classification SVM, SVR tries to find a function that deviates from the actual targets by at most ε for each training example.

**The SVR Optimization Problem:**

Minimize:
$$\frac{1}{2}||w||^2 + C \sum_{i=1}^{n} (\xi_i + \xi_i^*)$$

Subject to:
$$y_i - (w^T x_i + b) \leq \epsilon + \xi_i$$
$$(w^T x_i + b) - y_i \leq \epsilon + \xi_i^*$$
$$\xi_i, \xi_i^* \geq 0$$

Where:
- **w** = weight vector
- **b** = bias term
- **ε (epsilon)** = margin of tolerance (tube width)
- **C** = regularization parameter (trade-off between margin and error)
- **ξᵢ, ξᵢ*** = slack variables (allow some points outside the margin)
- **xᵢ** = input feature vector
- **yᵢ** = target value

**Key Concepts:**
- **ε-tube**: A tube of width 2ε around the predicted function
- **Support Vectors**: Data points that lie outside the ε-tube
- **Kernel Trick**: Maps data to higher dimensions for non-linear relationships

**Common Kernels:**
- **Linear**: $K(x, x') = x^T x'$ (for linear relationships)
- **RBF (Gaussian)**: $K(x, x') = \exp(-\gamma ||x - x'||^2)$ (most popular)
- **Polynomial**: $K(x, x') = (x^T x' + c)^d$ (for polynomial relationships)

**Hyperparameters:**
- **C**: Regularization parameter (higher C = less regularization, more complex model)
- **ε (epsilon)**: Tube width (larger ε = more points inside tube, simpler model)
- **γ (gamma)**: Kernel coefficient (for RBF kernel, controls influence of single training point)

**Advantages:**
- Effective in high-dimensional spaces
- Versatile through different kernel functions
- Robust to overfitting with proper regularization

**Disadvantages:**
- Requires feature scaling
- Computationally intensive for large datasets
- Difficult to interpret
- Sensitive to noise

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train SVR with RBF Kernel (Default)

In [ ]:
svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)

svr.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = svr.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Model Information

In [ ]:
print("Number of Support Vectors:", svr.n_support_)
print("Total Support Vectors:", sum(svr.n_support_))

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "SVM Regression (RBF Kernel)"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Compare Different Kernels

Let's compare Linear, RBF, and Polynomial kernels.

In [ ]:
kernels = ['linear', 'rbf', 'poly']

results = []

for kernel in kernels:
    model = SVR(kernel=kernel, C=1.0, epsilon=0.1)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'Kernel': kernel,
        'R²': r2,
        'Support Vectors': sum(model.n_support_)
    })

results_df = pd.DataFrame(results)
print(results_df)

## Hyperparameter Tuning for RBF Kernel

Let's tune C and epsilon parameters for the RBF kernel.

In [ ]:
C_values = [0.1, 1, 10, 100]
epsilon_values = [0.01, 0.1, 0.5, 1.0]

results = []

for C in C_values:
    for epsilon in epsilon_values:
        model = SVR(kernel='rbf', C=C, epsilon=epsilon)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        r2 = r2_score(y_test, pred)
        
        results.append({
            'C': C,
            'Epsilon': epsilon,
            'R²': r2,
            'Support Vectors': sum(model.n_support_)
        })

results_df = pd.DataFrame(results)
print(results_df.sort_values('R²', ascending=False).head(10))

## Visualize Hyperparameter Performance

In [ ]:
# Pivot for heatmap
pivot_r2 = results_df.pivot('C', 'Epsilon', 'R²')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_r2, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('R² Score Heatmap (RBF Kernel)')
plt.xlabel('Epsilon')
plt.ylabel('C')
plt.show()

## Summary

SVM Regression provides:
- **Non-linear modeling**: Can capture complex non-linear relationships through kernels
- **Effective in high dimensions**: Works well with many features
- **Regularization control**: C parameter balances complexity and error
- **Versatile kernels**: Different kernels for different data patterns

**Key considerations:**
- Requires feature scaling (crucial for performance)
- Computationally intensive for large datasets
- Sensitive to hyperparameter tuning
- Less interpretable than linear models

**Best practices:**
- Always scale features before training
- Use cross-validation for hyperparameter tuning
- Start with RBF kernel (most versatile)
- Consider computational cost for large datasets